# Part 1 – Data Preparation (5%)

**Dataset:** FER-2013 (publicly available benchmark for facial emotion recognition)  
**Selected classes:** Angry, Happy, Sad, Neutral  
**Source folder:** `Dataset/dataset/`

## Setup – Imports & Configuration

In [19]:
import os
import hashlib
import shutil
import random
from pathlib import Path
from PIL import Image

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR    = Path("Dataset/dataset")
OUTPUT_DIR  = Path("Dataset/processed")   # cleaned + resized images will go here
CLASSES     = ["Angry", "Happy", "Sad", "Neutral"]
IMG_SIZE    = (512, 512)                   # target size: 512×512
SEED        = 42
random.seed(SEED)

# Split ratios
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.20
TEST_RATIO  = 0.10

print("Classes:", CLASSES)
print("Source :", BASE_DIR)
print("Output :", OUTPUT_DIR)

Classes: ['Angry', 'Happy', 'Sad', 'Neutral']
Source : Dataset/dataset
Output : Dataset/processed


## Step 4 – Remove Redundant (Duplicate) Images

Duplicates are detected using **MD5 hashing**: if two files produce the same hash, only the first is kept.

In [20]:
def get_md5(filepath):
    """Return the MD5 hash of a file's contents."""
    h = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def remove_duplicates(class_dir):
    """Remove duplicate images in a folder using MD5 hashing. Returns (kept, removed) counts."""
    seen_hashes = {}
    removed = 0
    for img_path in sorted(class_dir.iterdir()):
        if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp"}:
            continue
        file_hash = get_md5(img_path)
        if file_hash in seen_hashes:
            img_path.unlink()   # delete duplicate
            removed += 1
        else:
            seen_hashes[file_hash] = img_path
    kept = len(seen_hashes)
    return kept, removed

print(f"{'Class':<10} {'Before':>8} {'Removed':>9} {'After':>7}")
print("-" * 38)
for cls in CLASSES:
    cls_dir = BASE_DIR / cls
    before = sum(1 for f in cls_dir.iterdir() if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
    kept, removed = remove_duplicates(cls_dir)
    print(f"{cls:<10} {before:>8} {removed:>9} {kept:>7}")

Class        Before   Removed   After
--------------------------------------
Angry          1306         0    1306
Happy          3729         0    3729
Sad            3906         0    3906
Neutral        4014         0    4014


## Steps 3 & 5 – Split (70 / 20 / 10) and Resize to 512×512×3

For each class:
1. Take **1000 images** (capped from the full deduplicated set)
2. Shuffle with a fixed seed for reproducibility
3. Split → **700 train / 200 val / 100 test**
4. Resize to 512×512 RGB — `LANCZOS` for downsampling, `BICUBIC` for upsampling

In [ ]:
def split_and_resize(classes, src_dir, dst_dir, img_size, train_r, val_r, number_of_images=10, seed=42):
    random.seed(seed)
    summary = {}

    for cls in classes:
        images = sorted([
            p for p in (src_dir / cls).iterdir()
            if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
        ])
        random.shuffle(images)

        # Cap to number_of_images per class, then split
        images  = images[:number_of_images]
        n_train = int(number_of_images * train_r)   # 700
        n_val   = int(number_of_images * val_r)     # 200
        # test = remainder = 100

        splits = {
            "train": images[:n_train],
            "val"  : images[n_train: n_train + n_val],
            "test" : images[n_train + n_val:],
        }
        summary[cls] = {s: len(v) for s, v in splits.items()}

        for split, paths in splits.items():
            out_dir = dst_dir / split / cls
            out_dir.mkdir(parents=True, exist_ok=True)
            for p in paths:
                try:
                    img = Image.open(p).convert("RGB")
                    w, h = img.size
                    resample = Image.LANCZOS if (w > img_size[0] or h > img_size[1]) else Image.BICUBIC
                    img = img.resize(img_size, resample)
                    img.save(out_dir / (p.stem + ".jpg"), "JPEG")
                except Exception as e:
                    print(f"  [skip] {p.name}: {e}")

    return summary


summary = split_and_resize(
    CLASSES, BASE_DIR, OUTPUT_DIR, IMG_SIZE,
    TRAIN_RATIO, VAL_RATIO, number_of_images=10, seed=SEED
)

print(f"{'Class':<10} {'Train':>7} {'Val':>7} {'Test':>7} {'Total':>8}")
print("-" * 43)
for cls, counts in summary.items():
    print(f"{cls:<10} {counts['train']:>7} {counts['val']:>7} {counts['test']:>7} {sum(counts.values()):>8}")
print(f"\nProcessed images saved to: {OUTPUT_DIR.resolve()}")

Class        Train     Val    Test    Total
-------------------------------------------
Angry          909     260     131     1300
Happy          909     260     131     1300
Sad            909     260     131     1300
Neutral        909     260     131     1300

Processed images saved to: /Users/ramezhany/Uni/Sem 10/Deep Learning/Assignment/Dataset/processed


## Verification – Sample Images per Class

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Display one sample per class from the training split
fig, axes = plt.subplots(1, len(CLASSES), figsize=(16, 4))
fig.suptitle("Sample images after resizing to 512×512×3", fontsize=14)

for ax, cls in zip(axes, CLASSES):
    cls_train_dir = OUTPUT_DIR / "train" / cls
    sample = next(cls_train_dir.iterdir())   # pick first image
    img = mpimg.imread(sample)
    ax.imshow(img)
    ax.set_title(f"{cls}\n{img.shape}")
    ax.axis("off")

plt.tight_layout()
plt.show()

---
# First Model (35%) – CNN from Scratch

We build a full convolutional feature extractor **from scratch using only NumPy** (no deep learning libraries).  
The extracted features are then fed to **K-Means** for unsupervised clustering.

Architecture overview:
```
Input (512×512×3)
  └── ConvBlock 1: ConvLayer → PoolingLayer → ReLU  →  (255×255×5)
  └── ConvBlock 2: ConvLayer → PoolingLayer → ReLU  →  (126×126×5)
  └── ConvBlock 3: ConvLayer → PoolingLayer → ReLU  →  ( 62× 62×5)
  └── Flatten                                        →  (19220,)
  └── Linear downsample                             →  (128,)
  └── K-Means clustering                            →  class label
```

## Step 1 – ConvLayer (10%)

### What does convolution do?
A filter (kernel) slides over the input image one position at a time.  
At each position it **element-wise multiplies** the kernel with the overlapping patch and **sums** the result → one number.  
Repeating this across the whole image produces a **feature map**.  
With 5 filters we get 5 feature maps stacked → output shape `(H-2, W-2, 5)` for a 3×3 kernel with no padding.

### Two initialization methods
| Method | When to use |
|--------|-------------|
| `ConvLayer.random(num_filters, filter_size)` | Create random filter weights |
| `ConvLayer(filter_weights)` | Pass a pre-defined 3-D weight array |

### The 5 predefined filters and what they detect
| Filter | Name | Detects |
|--------|------|---------|
| (a) all-ones | Box / Average | General brightness, blurs image |
| (b) identity | Identity | Passes image unchanged |
| (c) Sobel-X | Horizontal gradient | **Vertical edges** |
| (d) Sobel-Y | Vertical gradient | **Horizontal edges** |
| (e) Laplacian | Sharpening | **All edges / fine details** |

### Efficiency note
Instead of three nested Python loops (very slow on 512×512 images),  
we use `numpy.lib.stride_tricks.sliding_window_view` to extract all patches at once  
and `np.einsum` to multiply-and-sum in a single vectorized call.

In [ ]:
import numpy as np

class ConvLayer:
    """
    A single convolutional layer implemented from scratch with NumPy.

    Each filter is applied to every input channel independently;
    the results are summed across channels to produce one feature map per filter.
    This is standard cross-correlation (no padding, stride = 1).

    Two ways to create a ConvLayer
    ─────────────────────────────
    1) Random weights:
           layer = ConvLayer.random(num_filters=5, filter_size=3)

    2) Pre-defined weights (3-D array of shape [num_filters, h, w]):
           layer = ConvLayer(my_filter_array)
    """

    # ── Initialization method 2: from a pre-defined weight matrix ─────────────
    def __init__(self, filter_weights):
        """
        Parameters
        ----------
        filter_weights : array-like, shape (num_filters, filter_h, filter_w)
        """
        self.filters = np.array(filter_weights, dtype=np.float64)  # (F, kH, kW)
        self.num_filters, self.filter_h, self.filter_w = self.filters.shape

    # ── Initialization method 1: random weights ────────────────────────────────
    @classmethod
    def random(cls, num_filters, filter_size):
        """
        Parameters
        ----------
        num_filters : int   number of filters
        filter_size : int   spatial size (filter_size × filter_size)
        """
        weights = np.random.randn(num_filters, filter_size, filter_size)
        return cls(weights)

    # ── Internal helper: convolve a single 2-D channel with a single kernel ────
    @staticmethod
    def _convolve2d(channel, kernel):
        """
        2-D valid cross-correlation using explicit for loops.

        For each valid position (i, j) in the output:
          - Extract the patch of the same size as the kernel
          - Multiply element-wise with the kernel
          - Sum all values → one output value

        Output shape: (H - kH + 1,  W - kW + 1)
        """
        H, W      = channel.shape
        kH, kW    = kernel.shape
        out_H     = H - kH + 1
        out_W     = W - kW + 1
        output    = np.zeros((out_H, out_W))

        for i in range(out_H):
            for j in range(out_W):
                # Extract the patch and compute dot product with kernel
                patch         = channel[i:i+kH, j:j+kW]   # shape (kH, kW)
                output[i, j]  = np.sum(patch * kernel)

        return output

    # ── Iterate filters over the entire input (called inside forward) ──────────
    def iterate_filters(self, x):
        """
        Slide every filter over every channel of x and collect feature maps.

        Parameters
        ----------
        x : ndarray, shape (H, W, C)

        Returns
        -------
        output : ndarray, shape (H - kH + 1, W - kW + 1, num_filters)
        """
        H, W, C = x.shape
        out_H = H - self.filter_h + 1
        out_W = W - self.filter_w + 1
        output = np.zeros((out_H, out_W, self.num_filters))

        for f_idx, kernel in enumerate(self.filters):
            # Apply the same 2-D kernel to each channel and accumulate
            feature_map = np.zeros((out_H, out_W))
            for c in range(C):
                feature_map += self._convolve2d(x[:, :, c], kernel)
            output[:, :, f_idx] = feature_map

        return output

    # ── Forward pass ───────────────────────────────────────────────────────────
    def forward(self, x):
        """
        Run a full forward pass through this convolutional layer.

        Parameters
        ----------
        x : ndarray, shape (H, W, C)   input image or feature map

        Returns
        -------
        ndarray, shape (H - kH + 1, W - kW + 1, num_filters)
        """
        return self.iterate_filters(x)


# ── Define the 5 predefined 3×3 filters from the assignment ───────────────────
PREDEFINED_FILTERS = np.array([
    # (a) Box filter – averages/blurs the image
    [[ 1,  1,  1],
     [ 1,  1,  1],
     [ 1,  1,  1]],

    # (b) Identity – passes the image unchanged
    [[ 0,  0,  0],
     [ 0,  1,  0],
     [ 0,  0,  0]],

    # (c) Sobel-X – detects vertical edges (horizontal gradient)
    [[-1,  0,  1],
     [-2,  0,  2],
     [-1,  0,  1]],

    # (d) Sobel-Y – detects horizontal edges (vertical gradient)
    [[-1, -2, -1],
     [ 0,  0,  0],
     [ 1,  2,  1]],

    # (e) Sharpening / Laplacian – enhances all edges and fine details
    [[ 0, -1,  0],
     [-1,  5, -1],
     [ 0, -1,  0]],
], dtype=np.float64)  # shape: (5, 3, 3)

# Initialize ConvLayer using the predefined filters (initialization method 2)
conv_layer = ConvLayer(PREDEFINED_FILTERS)

print("ConvLayer created with predefined filters.")
print(f"  Number of filters : {conv_layer.num_filters}")
print(f"  Filter shape      : {conv_layer.filter_h} × {conv_layer.filter_w}")
print(f"  Filters array shape: {conv_layer.filters.shape}")

# Quick sanity check: random ConvLayer (initialization method 1)
random_conv = ConvLayer.random(num_filters=5, filter_size=3)
print(f"\nRandom ConvLayer filters shape: {random_conv.filters.shape}")

## Step 2 – PoolingLayer (5%)

### What does pooling do?
After convolution the feature maps are still large (e.g. 510×510).  
Pooling **shrinks** them by scanning a small window (default **2×2**) across the map and replacing each window with a **single value**:

| Type | Formula | Effect |
|------|---------|--------|
| **MAX** | take the largest value in the window | keeps the strongest activation — most common |
| **AVERAGE** | take the mean of all values | smoother output |

- Stride = pool size → windows are **non-overlapping**
- No weights, no learning — purely structural
- 510×510 with a 2×2 pool → **255×255** (halved in both dimensions)
- Applied independently to each feature map (each of the 5 channels)

In [ ]:
class PoolingLayer:
    """
    Pooling layer implemented from scratch with NumPy.

    Slides a (pool_size × pool_size) window over each channel of the input
    with stride = pool_size (non-overlapping windows) and reduces each window
    to a single value using either MAX or AVERAGE.

    Usage
    -----
    pool = PoolingLayer()                        # 2×2 MAX (defaults)
    pool = PoolingLayer(pool_size=2, pool_type='AVERAGE')
    output = pool.forward(x)                     # x shape: (H, W, C)
    """

    def __init__(self, pool_size=2, pool_type='MAX'):
        """
        Parameters
        ----------
        pool_size : int     side length of the square pooling window (default 2)
        pool_type : str     'MAX' or 'AVERAGE' (default 'MAX')
        """
        assert pool_type in ('MAX', 'AVERAGE'), "pool_type must be 'MAX' or 'AVERAGE'"
        self.pool_size = pool_size
        self.pool_type = pool_type

    # ── Iterate the pooling window over one feature map (single channel) ───────
    def _pool_channel(self, channel):
        """
        Apply pooling to a single 2-D feature map using explicit for loops.

        The window moves by pool_size each step (stride = pool_size),
        so windows never overlap.  Any incomplete border rows/columns are ignored
        (equivalent to 'valid' mode).

        Parameters
        ----------
        channel : ndarray, shape (H, W)

        Returns
        -------
        ndarray, shape (H // pool_size, W // pool_size)
        """
        H, W   = channel.shape
        p      = self.pool_size
        out_H  = H // p
        out_W  = W // p
        output = np.zeros((out_H, out_W))

        for i in range(out_H):
            for j in range(out_W):
                # Extract the current pool window
                window = channel[i*p : i*p+p,
                                 j*p : j*p+p]   # shape (p, p)

                if self.pool_type == 'MAX':
                    output[i, j] = np.max(window)
                else:                            # AVERAGE
                    output[i, j] = np.mean(window)

        return output

    # ── Iterate over all channels (called inside forward) ──────────────────────
    def iterate_filters(self, x):
        """
        Apply pooling independently to every channel of the input.

        Parameters
        ----------
        x : ndarray, shape (H, W, C)

        Returns
        -------
        ndarray, shape (H // pool_size, W // pool_size, C)
        """
        H, W, C = x.shape
        out_H   = H // self.pool_size
        out_W   = W // self.pool_size
        output  = np.zeros((out_H, out_W, C))

        for c in range(C):
            output[:, :, c] = self._pool_channel(x[:, :, c])

        return output

    # ── Forward pass ───────────────────────────────────────────────────────────
    def forward(self, x):
        """
        Run a full forward pass through this pooling layer.

        Parameters
        ----------
        x : ndarray, shape (H, W, C)

        Returns
        -------
        ndarray, shape (H // pool_size, W // pool_size, C)
        """
        return self.iterate_filters(x)


# ── Sanity check ───────────────────────────────────────────────────────────────
pool_layer = PoolingLayer(pool_size=2, pool_type='MAX')

# Simulate the output of ConvLayer: (510, 510, 5)
dummy_conv_output = np.random.rand(510, 510, 5)
pool_output = pool_layer.forward(dummy_conv_output)

print(f"PoolingLayer created  →  type: {pool_layer.pool_type}, size: {pool_layer.pool_size}×{pool_layer.pool_size}")
print(f"Input  shape : {dummy_conv_output.shape}")
print(f"Output shape : {pool_output.shape}   (510÷2=255)")

## Step 3 – Activation Function (2%)

The assignment defines the activation function as:

$$\sigma(z) = \begin{cases} z & z \geq 0 \\ 0 & \text{otherwise} \end{cases}$$

This is **ReLU** (Rectified Linear Unit). It:
- Keeps all **positive** values unchanged
- Sets all **negative** values to **zero**

**Why do we need it?**  
Convolution and pooling are both linear operations. Without an activation function, stacking multiple layers is mathematically equivalent to just one layer. ReLU introduces **non-linearity**, allowing the network to learn complex patterns.

Applied element-wise to the entire feature map — shape stays the same.

In [ ]:
def relu(z):
    """
    ReLU activation function applied element-wise.

    σ(z) = z  if z >= 0
           0  otherwise

    Parameters
    ----------
    z : ndarray, any shape

    Returns
    -------
    ndarray, same shape as z — negative values replaced with 0
    """
    return np.maximum(0, z)


# ── Sanity check ───────────────────────────────────────────────────────────────
# Simulate the output of PoolingLayer: (255, 255, 5)
dummy_pool_output = np.random.randn(255, 255, 5)  # randn gives negative values too

activated = relu(dummy_pool_output)

print(f"Input  shape  : {dummy_pool_output.shape}")
print(f"Output shape  : {activated.shape}   (unchanged)")
print(f"Min before ReLU : {dummy_pool_output.min():.4f}  →  after: {activated.min():.4f}")
print(f"Negative values before: {(dummy_pool_output < 0).sum()}  →  after: {(activated < 0).sum()}")

## Step 4 – Full Network: 3 ConvBlocks + Flatten + Downsample (8%)

### Architecture

Each **ConvBlock** = `ConvLayer → PoolingLayer → ReLU`

```
Input  (512 × 512 × 3)
  │
  ├── ConvBlock 1 ──────────────────────────────────────────────
  │     ConvLayer  (5 filters, 3×3)  →  (510, 510, 5)
  │     PoolingLayer (2×2 MAX)        →  (255, 255, 5)
  │     ReLU                          →  (255, 255, 5)
  │
  ├── ConvBlock 2 ──────────────────────────────────────────────
  │     ConvLayer  (5 filters, 3×3)  →  (253, 253, 5)
  │     PoolingLayer (2×2 MAX)        →  (126, 126, 5)
  │     ReLU                          →  (126, 126, 5)
  │
  ├── ConvBlock 3 ──────────────────────────────────────────────
  │     ConvLayer  (5 filters, 3×3)  →  (124, 124, 5)
  │     PoolingLayer (2×2 MAX)        →  ( 62,  62, 5)
  │     ReLU                          →  ( 62,  62, 5)
  │
  ├── Flatten  →  (19220,)        [62 × 62 × 5 = 19220]
  │
  └── Linear Downsample  W(19220 × 128)  →  (128,)   ← feature vector
```

### Downsample to 128
A fixed random weight matrix **W** of shape **(19220 × 128)** is used.  
`feature = flatten @ W` — this is a single matrix multiplication, projecting the 19220-dim vector down to 128 dimensions.  
These 128 values are the **feature vector** for one image, fed to K-Means.

In [ ]:
class CNNFromScratch:
    """
    Feature extractor built from scratch using only NumPy.

    Architecture:
        ConvBlock 1  →  ConvBlock 2  →  ConvBlock 3
        →  Flatten  →  Downsample to 128 (average pooling over groups)

    Each ConvBlock = ConvLayer → PoolingLayer → ReLU.
    The same 5 predefined filters are reused in every ConvLayer.
    Downsampling splits the flat vector into 128 equal groups and averages each.
    """

    def __init__(self, filters, pool_size=2, pool_type='MAX', feature_dim=128):
        """
        Parameters
        ----------
        filters    : ndarray (num_filters, kH, kW)  predefined filter weights
        pool_size  : int     pooling window size (default 2)
        pool_type  : str     'MAX' or 'AVERAGE' (default 'MAX')
        feature_dim: int     output feature vector size (default 128)
        """
        # ── 3 ConvBlocks ───────────────────────────────────────────────────────
        self.conv1 = ConvLayer(filters)
        self.conv2 = ConvLayer(filters)
        self.conv3 = ConvLayer(filters)

        self.pool1 = PoolingLayer(pool_size, pool_type)
        self.pool2 = PoolingLayer(pool_size, pool_type)
        self.pool3 = PoolingLayer(pool_size, pool_type)

        self.feature_dim = feature_dim

    # ── Single ConvBlock: Conv → Pool → ReLU ───────────────────────────────────
    def _conv_block(self, x, conv, pool):
        x = conv.forward(x)   # convolution
        x = pool.forward(x)   # pooling
        x = relu(x)           # activation
        return x

    # ── Downsample flat vector to feature_dim by average pooling ───────────────
    def _downsample(self, flat, target_size):
        """
        Split the flat vector into target_size equal groups,
        return the mean of each group.

        e.g. flat shape (19220,) → split into 128 groups of ~150 values each
             → take mean of each group → output shape (128,)

        Parameters
        ----------
        flat        : ndarray, shape (N,)
        target_size : int   number of output values (128)

        Returns
        -------
        ndarray, shape (target_size,)
        """
        N      = len(flat)
        # Trim flat so it divides evenly into target_size groups
        trim   = N - (N % target_size)        # drop the last few values if needed
        flat   = flat[:trim]
        # Reshape into (target_size, group_size) then average each row
        groups = flat.reshape(target_size, -1) # (128, group_size)
        return groups.mean(axis=1)             # (128,)

    # ── Forward pass ───────────────────────────────────────────────────────────
    def forward(self, x):
        """
        Extract a 128-dim feature vector from one image.

        Parameters
        ----------
        x : ndarray, shape (H, W, C)   — normalised image (values 0–1)

        Returns
        -------
        feature : ndarray, shape (128,)
        """
        # ── 3 ConvBlocks ───────────────────────────────────────────────────────
        x = self._conv_block(x, self.conv1, self.pool1)  # (255, 255, 5)
        x = self._conv_block(x, self.conv2, self.pool2)  # (126, 126, 5)
        x = self._conv_block(x, self.conv3, self.pool3)  # ( 62,  62, 5)

        # ── Flatten ────────────────────────────────────────────────────────────
        flat = x.flatten()                               # (19220,)

        # ── Downsample: 19220 → 128 via group averaging ────────────────────────
        feature = self._downsample(flat, self.feature_dim)  # (128,)

        return feature

    # ── Run on a list of image paths ───────────────────────────────────────────
    def extract_features(self, image_paths):
        """
        Run the full network on a list of image paths.

        Parameters
        ----------
        image_paths : list of Path objects

        Returns
        -------
        features : ndarray, shape (N, 128)
        labels   : list of str  — class name for each image (folder name)
        """
        features = []
        labels   = []

        for idx, path in enumerate(image_paths):
            img  = np.array(Image.open(path).convert('RGB'), dtype=np.float64) / 255.0
            feat = self.forward(img)
            features.append(feat)
            labels.append(path.parent.name)

            if (idx + 1) % 10 == 0:
                print(f"  Processed {idx + 1} / {len(image_paths)}")

        return np.array(features), labels


# ── Build the network ──────────────────────────────────────────────────────────
cnn = CNNFromScratch(filters=PREDEFINED_FILTERS, pool_size=2, pool_type='MAX', feature_dim=128)

print("CNNFromScratch built successfully.")
print(f"  ConvBlocks  : 3  (Conv → Pool → ReLU)")
print(f"  Filters     : {PREDEFINED_FILTERS.shape[0]} predefined 3×3 filters")
print(f"  Pooling     : 2×2 MAX")
print(f"  Downsample  : group averaging  19220 → 128")

# ── Dimension test on one dummy image ─────────────────────────────────────────
print("\nRunning dimension test on a single 512×512×3 dummy image...")
dummy_img = np.random.rand(512, 512, 3)
feat = cnn.forward(dummy_img)
print(f"  Input  shape : (512, 512, 3)")
print(f"  Output shape : {feat.shape}   ✓")

## Step 5 – K-Means Clustering (2%)

### What is K-Means?
K-Means groups N feature vectors into K clusters **without using labels** (unsupervised).

**Algorithm:**
1. Pick K random points from the data as initial **centroids**
2. **Assign** each point to the nearest centroid (Euclidean distance)
3. **Update** each centroid to be the mean of all points assigned to it
4. Repeat steps 2–3 until centroids stop moving (convergence)

We use **K = 4** (one cluster per emotion class).

### Flow
```
10 images × 4 classes = 40 images
       ↓  cnn.extract_features()
feature matrix  (40, 128)
       ↓  KMeans(k=4)
cluster labels  (40,)   ← 0,1,2,3
       ↓  compare with true labels
accuracy
```

In [ ]:
from sklearn.cluster import KMeans
from itertools import permutations

IMAGES_PER_CLASS = 7

# ── Extract train features (fit K-Means on these) ─────────────────────────────
train_paths = []
for cls in CLASSES:
    paths = sorted((OUTPUT_DIR / "train" / cls).iterdir())[:IMAGES_PER_CLASS]
    train_paths.extend(paths)

print(f"Extracting TRAIN features  ({IMAGES_PER_CLASS} per class = {len(train_paths)} total)...")
train_features, train_labels = cnn.extract_features(train_paths)
print(f"Train feature matrix shape : {train_features.shape}")

# ── Extract test features (evaluate on these) ─────────────────────────────────
test_paths = []
for cls in CLASSES:
    paths = sorted((OUTPUT_DIR / "test" / cls).iterdir())[:IMAGES_PER_CLASS]
    test_paths.extend(paths)

print(f"\nExtracting TEST features  ({IMAGES_PER_CLASS} per class = {len(test_paths)} total)...")
test_features, test_labels = cnn.extract_features(test_paths)
print(f"Test feature matrix shape  : {test_features.shape}")

In [ ]:
# ── Fit K-Means on train features ─────────────────────────────────────────────
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans.fit(train_features)

train_cluster_labels = kmeans.predict(train_features)
train_true_numeric   = np.array([CLASSES.index(l) for l in train_labels])

# ── Majority vote: for each class, which cluster did most of its images land in?
class_to_cluster = {}

for class_id in range(4):

    count_cluster_0 = 0
    count_cluster_1 = 0
    count_cluster_2 = 0
    count_cluster_3 = 0

    for i in range(len(train_true_numeric)):
        if train_true_numeric[i] == class_id:
            if train_cluster_labels[i] == 0:
                count_cluster_0 = count_cluster_0 + 1
            elif train_cluster_labels[i] == 1:
                count_cluster_1 = count_cluster_1 + 1
            elif train_cluster_labels[i] == 2:
                count_cluster_2 = count_cluster_2 + 1
            elif train_cluster_labels[i] == 3:
                count_cluster_3 = count_cluster_3 + 1

    # find which cluster has the highest count
    best_cluster = 0
    best_count   = count_cluster_0

    if count_cluster_1 > best_count:
        best_count   = count_cluster_1
        best_cluster = 1
    if count_cluster_2 > best_count:
        best_count   = count_cluster_2
        best_cluster = 2
    if count_cluster_3 > best_count:
        best_count   = count_cluster_3
        best_cluster = 3

    class_to_cluster[class_id] = best_cluster

print("Class → Cluster mapping:")
for class_id in range(4):
    print(f"  {CLASSES[class_id]}  →  Cluster {class_to_cluster[class_id]}")

# ── Invert: for prediction we need cluster → class ────────────────────────────
cluster_to_class = {}
for class_id in range(4):
    cluster_id = class_to_cluster[class_id]
    cluster_to_class[cluster_id] = class_id

print("\nCluster → Class mapping:")
for cluster_id in range(4):
    print(f"  Cluster {cluster_id}  →  {CLASSES[cluster_to_class[cluster_id]]}")

# ── Predict on TEST set ────────────────────────────────────────────────────────
test_cluster_labels = kmeans.predict(test_features)
print(f"\nTest cluster assignments : {list(test_cluster_labels)}")
print(f"Test true labels         : {[CLASSES.index(l) for l in test_labels]}")

In [ ]:
# ── Evaluate on TEST set ──────────────────────────────────────────────────────
test_true_numeric = np.array([CLASSES.index(l) for l in test_labels])

# Convert predicted cluster IDs → class IDs using the inverted mapping
test_mapped = np.array([cluster_to_class[c] for c in test_cluster_labels])
test_acc    = np.mean(test_mapped == test_true_numeric)

print(f"K-Means Test Accuracy : {test_acc * 100:.1f}%  ({int(test_acc * len(test_true_numeric))}/{len(test_true_numeric)} correct)")